# DQN baseline on Atari

In [1]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import time
import math
import imageio
import gymnasium as gym
import ale_py
from IPython.display import Image, display


# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.atari_envs import make_atari_goal_env
from utils import AtariReplayBuffer, evaluate_policy
from visualisations import plot_policy_rollouts, plot_q_diagnostics
from networks import DQN_Atari_CNN
import random


DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)

OUTPUT_DIR = repo_root / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


Using device: mps


In [2]:
# Explicitly register ALE envs (required for gymnasium>=1.0)
gym.register_envs(ale_py)
print("Registered ALE environments.")

env = gym.make("ALE/Breakout-v5", render_mode="rgb_array")
obs, info = env.reset()
print("obs.shape:", obs.shape)
print("action_space:", env.action_space)
env.close()

Registered ALE environments.
obs.shape: (210, 160, 3)
action_space: Discrete(4)


A.L.E: Arcade Learning Environment (version 0.12.0+unknown)
[Powered by Stella]


In [3]:
ENV_ID = "ALE/Breakout-v5"
FRAME_STACK = 4
MAX_HORIZON = 27_000
SEED = 42

def make_env(seed=None, max_horizon=MAX_HORIZON, render_mode=None):
    env = make_atari_goal_env(
        env_id=ENV_ID,
        frame_stack=FRAME_STACK,
        goal_obs=None,
        reward_mode="simple",
        goal_radius=0.0,
        max_episode_steps=max_horizon,
        flatten_obs=False,
    )
    if seed is not None:
        env.reset(seed=seed)
    return env

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
env = make_env(seed=SEED)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("obs_shape:", env.observation_space.shape)
print("num_actions:", env.action_space.n)
env.close()

Observation space: Box(0.0, 255.0, (4, 84, 84), float32)
Action space: Discrete(4)
obs_shape: (4, 84, 84)
num_actions: 4


In [4]:
BUFFER_CAPACITY = 500000
LR = float(1e-4)
GAMMA = 0.99
BATCH_SIZE = 64
TOTAL_STEPS = 1000000
WARMUP_STEPS = 50000
TRAIN_FREQ = 4
TARGET_UPDATE_FREQ = 10000
EPS_START = 1.0
EPS_END = 0.1
EPS_DECAY_STEPS = 250_000

env_tmp = make_env(seed=SEED)
obs_shape = env_tmp.observation_space.shape
num_actions = env_tmp.action_space.n
env_tmp.close()

q_net = DQN_Atari_CNN(obs_shape, num_actions).to(DEVICE)
q_target = DQN_Atari_CNN(obs_shape, num_actions).to(DEVICE)
q_target.load_state_dict(q_net.state_dict())

for p in q_target.parameters():
    p.requires_grad_(False)

print("obs_shape =", obs_shape)
print("num_actions =", num_actions)


def greedy_action(q_network, obs, device=DEVICE):
    obs_t = torch.as_tensor(obs, dtype=torch.float32, device=device)
    if obs_t.ndim == 3:
        obs_t = obs_t.unsqueeze(0)
    if obs_t.max() > 1.5:
        obs_t = obs_t / 255.0
    with torch.no_grad():
        action = int(q_network(obs_t).argmax(dim=-1).item())
    return action

def dqn_train_atari(
    q_network,
    q_target_network,
    make_env_fn,
    seed=42,
    buffer_capacity=500_000,
    lr=1e-4,
    obs_shape=(4, 84, 84),
    device="cpu",
    total_steps=1_000_000,
    warmup_steps=50_000,
    batch_size=64,
    gamma=0.99,
    eps_start=1.0,
    eps_end=0.1,
    eps_decay_steps=250_000,
    train_freq=4,
    target_update_freq=10_000,
    eval_freq=50_000,
    eval_episodes=5,
):
    set_seed(seed)
    env = make_env_fn(seed=seed)
    opt = optim.Adam(q_network.parameters(), lr=lr)
    buffer = AtariReplayBuffer(buffer_capacity, obs_shape, device=device)

    obs, _ = env.reset(seed=seed)
    global_step = 0
    eval_returns = []
    train_losses = []
    wall_clock_start = time.perf_counter()

    while global_step < total_steps:
        frac = min(1.0, global_step / eps_decay_steps)
        eps = eps_start + frac * (eps_end - eps_start)

        if np.random.random() < eps:
            action = env.action_space.sample()
        else:
            action = greedy_action(q_network, obs, device=device)

        next_obs, rew, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        buffer.add_transition(
            obs=obs,
            action=action,
            reward=rew,
            next_obs=next_obs,
            terminated=terminated,
            truncated=truncated,
        )

        obs = next_obs
        global_step += 1

        if done:
            obs, _ = env.reset()

        if len(buffer) >= warmup_steps and global_step % train_freq == 0:
            batch = buffer.sample(batch_size)

            obs_t = batch.obs.float()
            act_t = batch.actions.long()
            rew_t = batch.rewards.float()
            next_obs_t = batch.next_obs.float()
            term_t = batch.terminated.float()

            with torch.no_grad():
                next_q = q_target_network(next_obs_t).max(dim=-1, keepdim=True).values
                target = rew_t + gamma * (1.0 - term_t) * next_q

            current_q = q_network(obs_t).gather(1, act_t.unsqueeze(1))
            loss = F.mse_loss(current_q, target)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_network.parameters(), 10.0)
            opt.step()

            train_losses.append((global_step, float(loss.item())))

        if global_step >= warmup_steps and global_step % target_update_freq == 0:
            q_target_network.load_state_dict(q_network.state_dict())

        if global_step % eval_freq == 0:
            eval_env = make_env_fn(seed=seed + 123)
            returns = []
            lengths = []
            for _ in range(eval_episodes):
                eobs, _ = eval_env.reset()
                done = False
                ep_ret = 0.0
                ep_len = 0
                while not done:
                    action = greedy_action(q_network, eobs, device=device)
                    eobs, rew, terminated, truncated, _ = eval_env.step(action)
                    done = terminated or truncated
                    ep_ret += float(rew)
                    ep_len += 1
                returns.append(ep_ret)
                lengths.append(ep_len)
            mean_ret = float(np.mean(returns))
            mean_len = float(np.mean(lengths))
            eval_returns.append((global_step, mean_ret))
            print(f"[eval] step={global_step:7d} | eps={eps:.3f} | eval_return={mean_ret:.3f} | eval_len={mean_len:.1f}")
            eval_env.close()

    total_elapsed = time.perf_counter() - wall_clock_start
    print(f"Total training wall-clock time: {total_elapsed:.1f} s")
    env.close()
    return q_network, q_target_network, eval_returns, train_losses, buffer


obs_shape = (4, 84, 84)
num_actions = 4


In [5]:
q_net, q_target, eval_returns, train_losses, buffer = dqn_train_atari(
    q_network=q_net,
    q_target_network=q_target,
    make_env_fn=make_env,
    seed=SEED,
    buffer_capacity=BUFFER_CAPACITY,
    lr=LR,
    obs_shape=obs_shape,
    device=DEVICE,
    total_steps=TOTAL_STEPS,
    warmup_steps=WARMUP_STEPS,
    batch_size=BATCH_SIZE,
    gamma=GAMMA,
    eps_start=EPS_START,
    eps_end=EPS_END,
    eps_decay_steps=EPS_DECAY_STEPS,
    train_freq=TRAIN_FREQ,
    target_update_freq=TARGET_UPDATE_FREQ,
    eval_freq=50_000,
    eval_episodes=5,
)

if eval_returns:
    xs, ys = zip(*eval_returns)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys, marker="o")
    plt.xlabel("Environment steps")
    plt.ylabel("Mean episodic return")
    plt.title(f"DQN baseline on {ENV_ID}")
    plt.grid(alpha=0.25)
    plt.show()

if train_losses:
    xs, ys = zip(*train_losses)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys, alpha=0.9)
    plt.xlabel("Environment steps")
    plt.ylabel("TD loss")
    plt.title(f"DQN training loss on {ENV_ID}")
    plt.grid(alpha=0.25)
    plt.show()

[eval] step=  50000 | eps=0.820 | eval_return=0.000 | eval_len=26988.4
[eval] step= 100000 | eps=0.640 | eval_return=0.000 | eval_len=26988.4
[eval] step= 150000 | eps=0.460 | eval_return=1.400 | eval_len=16265.2


KeyboardInterrupt: 

## Visualisations

In [ ]:
def collect_episode_frames(q_network, env, max_steps=5000, device=DEVICE):
    frames = []
    obs, info = env.reset()
    done = False
    step = 0
    total_reward = 0.0

    while not done and step < max_steps:
        frame = env.render()
        if frame is not None:
            frames.append(np.asarray(frame))

        action = greedy_action(q_network, obs, device=device)
        obs, reward, term, trunc, info = env.step(action)
        total_reward += float(reward)
        done = term or trunc
        step += 1

    frame = env.render()
    if frame is not None:
        frames.append(np.asarray(frame))

    return frames, total_reward, step

eval_env_frames = make_env(seed=SEED + 999)
frames, ep_return, ep_len = collect_episode_frames(dqn_q, eval_env_frames, max_steps=5000)
eval_env_frames.close()

print("Collected frames:", len(frames))
print("Episode return:", ep_return)
print("Episode length:", ep_len)

def show_sampled_frames(frames, n=12, figsize=(16, 8)):
    if len(frames) == 0:
        print("No frames collected.")
        return

    idxs = np.linspace(0, len(frames) - 1, min(n, len(frames)), dtype=int)
    n_show = len(idxs)
    n_cols = 4
    n_rows = math.ceil(n_show / n_cols)

    plt.figure(figsize=figsize)
    for i, idx in enumerate(idxs):
        plt.subplot(n_rows, n_cols, i + 1)
        plt.imshow(frames[idx])
        plt.title(f"frame {idx}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_sampled_frames(frames, n=12)


gif_path = OUTPUT_DIR / "dqn_atari_eval.gif"

if len(frames) > 0:
    imageio.mimsave(gif_path, frames, fps=12)
    print("Saved GIF to:", gif_path)
    display(Image(filename=str(gif_path)))
else:
    print("No frames available for GIF export.")

In [ ]:
def evaluate_and_save_gif(
    q_network,
    make_env_fn,
    gif_name="atari_eval.gif",
    seed=0,
    max_steps=5000,
    fps=12,
    device=DEVICE,
):
    env = make_env_fn(seed=seed)
    frames, ep_return, ep_len = collect_episode_frames(
        q_network=q_network,
        env=env,
        max_steps=max_steps,
        device=device,
    )
    env.close()

    gif_path = OUTPUT_DIR / gif_name
    if len(frames) > 0:
        imageio.mimsave(gif_path, frames, fps=fps)

    print("Episode return:", ep_return)
    print("Episode length:", ep_len)
    print("Num frames:", len(frames))
    print("GIF path:", gif_path)

    return frames, ep_return, ep_len, gif_path

frames_eval, ret_eval, len_eval, gif_path = evaluate_and_save_gif(
    q_network=dqn_q,
    make_env_fn=make_env,
    gif_name="dqn_breakout_eval.gif",
    seed=SEED + 2024,
    max_steps=5000,
    fps=12,
)

display(Image(filename=str(gif_path)))

## Now to check how much samples needed to reach a new goal